# Breast cancer classification with scikit-learn and OMLE

Train a `StandardScaler` → `LogisticRegression` pipeline, export the **entire fitted pipeline**, and verify OMLE predictions on the held-out test set.

This is a format and interoperability example, not a clinically validated diagnostic model.

## Dataset

The Breast Cancer Wisconsin (Diagnostic) dataset contains 569 samples and 30 numeric features computed from digitized fine-needle aspirate images. scikit-learn bundles the dataset, so no download is required.

- [scikit-learn loader and description](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)
- [Original UCI dataset, attribution and CC BY 4.0 license](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)
- Citation: Wolberg, W., Mangasarian, O., Street, N., & Street, W. (1993). Breast Cancer Wisconsin (Diagnostic). UCI Machine Learning Repository. https://doi.org/10.24432/C5DW2B.

Labels are **0 = malignant, 1 = benign**. ROC AUC below explicitly treats malignant as the positive condition.

Run all cells in order using the repository requirements. No separate training script is needed.

In [ ]:
import json
import platform
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from omle_convert import export_omle
import omle_runtime as omr

SEED = 42
ATOL = 1e-4
RTOL = 1e-4
packages = ["numpy", "pandas", "scikit-learn", "omle", "omle-convert", "omle-runtime", "omle-viewer"]
environment = {"python": platform.python_version(), "packages": {p: version(p) for p in packages}}
display(environment)

## Load and split

Use a stratified 80/20 split with a fixed seed. Preserve feature names and their order. The test set is used for final evaluation and conversion verification, not fitting or tuning.

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X, y = dataset.data, dataset.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
display(pd.DataFrame({
    "class": dataset.target_names,
    "all": np.bincount(y, minlength=2),
    "train": np.bincount(y_train, minlength=2),
    "test": np.bincount(y_test, minlength=2),
}))
print(f"Features: {X.shape[1]}; training rows: {len(X_train)}; test rows: {len(X_test)}")
display(X_train.head())

## Train and evaluate

Scaling is fitted inside the pipeline using training data only. Probability columns follow `pipeline.classes_`: malignant first, then benign.

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000, random_state=SEED)),
])
pipeline.fit(X_train, y_train)
native_labels = pipeline.predict(X_test)
native_probabilities = pipeline.predict_proba(X_test)
np.testing.assert_array_equal(pipeline.classes_, [0, 1])
malignant_column = list(pipeline.classes_).index(0)
metrics = {
    "accuracy": float(accuracy_score(y_test, native_labels)),
    "log_loss": float(log_loss(y_test, native_probabilities, labels=pipeline.classes_)),
    "roc_auc_malignant": float(roc_auc_score(
        (y_test == 0).astype(int), native_probabilities[:, malignant_column]
    )),
}
display(pd.Series(metrics, name="held_out_metrics"))
print(classification_report(y_test, native_labels, target_names=dataset.target_names))

## Export the fitted pipeline

Locate the repository from either its root or a notebook subdirectory. Export the scaler and classifier together. Embedded verification rows come from training data; the full held-out test set is checked independently below. Rerunning overwrites this example’s generated files.

In [ ]:
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents]
                  if (p / "requirements.txt").is_file() and (p / "notebooks").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Start Jupyter from the omle-models repository or one of its subdirectories.")
output_dir = repo_root / "models" / "classification" / "breast-cancer"
output_dir.mkdir(parents=True, exist_ok=True)
model_path = output_dir / "sklearn-logistic-regression.omle"
export_omle(
    pipeline, str(model_path), X=X_train,
    target_name="diagnosis", model_name="breast-cancer-sklearn-logistic-regression",
    model_version="0.1.0", random_state=SEED,
    n_verify=10, n_sample=3, verify_atol=ATOL, verify_rtol=RTOL,
)
print(f"Exported {model_path.relative_to(repo_root)} ({model_path.stat().st_size:,} bytes)")

## Inspect the exported model

Load the saved artifact and inspect its declared inputs and outputs: names, tensor types, shapes, and output roles. A shape dimension of `-1` represents a variable batch size. These declarations describe the model interface; probability columns follow the class ordering checked above.

Next, validate the model structure and operator declarations with `omle.validate`. Validation errors stop execution before the viewer is shown. Structural validation complements the numerical prediction parity check in the following section.

In [ ]:
import omle

exported_model = omle.load(str(model_path))

print("Model inputs")
display(pd.DataFrame([
    {
        "name": spec.name,
        "dtype": spec.type.dtype.name if spec.type else None,
        "shape": spec.type.shape if spec.type else None,
    }
    for spec in exported_model.inputs
]))

print("Model outputs")
display(pd.DataFrame([
    {
        "name": spec.name,
        "dtype": spec.type.dtype.name if spec.type else None,
        "shape": spec.type.shape if spec.type else None,
        "role": spec.role.name,
    }
    for spec in exported_model.outputs
]))

### Operators included in the model

List each graph node with its operator namespace and imported version. The fitted scaler becomes `omle.feature.StandardScaler`; logistic regression becomes `omle.ml.Linear`.

In [ ]:
operator_versions = {
    item.namespace: item.version for item in exported_model.operator_imports
}
operators = pd.DataFrame([
    {
        "node": node.name,
        "namespace": node.domain,
        "operator": node.op,
        "version": operator_versions.get(node.domain),
    }
    for node in exported_model.nodes
])
display(operators)

In [ ]:
validation = omle.validate(exported_model)
print(validation)
if not validation.is_valid:
    raise ValueError(f"Exported OMLE model is invalid:\n{validation}")

### Explore the interactive graph

Select the scaler and classifier nodes to inspect their fitted parameters and connections. The viewer is embedded locally; no model upload or external viewer server is needed.

Run the cell below in JupyterLab or your IDE notebook editor to interact with the graph. GitHub’s static notebook preview will not run it.

In [ ]:
from omle_viewer import show

viewer = show(exported_model, height=650)

## Verify OMLE inference

Pass raw, unscaled features in their original order: the OMLE graph includes scaling. Require exact label agreement and numeric probability agreement at the stated tolerances, including shapes and probability normalization. Quality metrics above and conversion fidelity below answer different questions.

In [ ]:
runtime_model = omr.load(str(model_path))
omle_labels = runtime_model.predict(X_test)
omle_probabilities = runtime_model.predict_proba(X_test)
assert omle_labels.shape == native_labels.shape
assert omle_probabilities.shape == native_probabilities.shape
assert np.isfinite(omle_probabilities).all()
assert ((omle_probabilities >= 0) & (omle_probabilities <= 1)).all()
np.testing.assert_array_equal(omle_labels, native_labels)
np.testing.assert_allclose(omle_probabilities, native_probabilities, atol=ATOL, rtol=RTOL)
np.testing.assert_allclose(omle_probabilities.sum(axis=1), 1.0, atol=ATOL, rtol=RTOL)
parity = {
    "test_rows": len(X_test),
    "label_agreement": float(np.mean(omle_labels == native_labels)),
    "max_absolute_probability_error": float(np.max(np.abs(omle_probabilities - native_probabilities))),
    "atol": ATOL, "rtol": RTOL,
    "native_label_dtype": str(native_labels.dtype),
    "omle_label_dtype": str(omle_labels.dtype),
    "native_probability_dtype": str(native_probabilities.dtype),
    "omle_probability_dtype": str(omle_probabilities.dtype),
}
display(pd.Series(parity, name="conversion_fidelity"))
print("PASS: OMLE labels and probabilities match the native pipeline.")

## Save the run record

Record the environment, feature and class ordering, split membership, and results alongside the generated model. Both files are ignored by Git until explicitly selected for publication. Model metrics may change with dependency versions; a fixed seed alone does not guarantee identical artifacts across versions.

In [ ]:
report = {
    "environment": environment,
    "dataset": "sklearn.datasets.load_breast_cancer",
    "dataset_source": "https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic",
    "seed": SEED, "test_size": 0.2,
    "feature_names": list(X.columns),
    "class_names": {str(i): str(name) for i, name in enumerate(dataset.target_names)},
    "train_indices": X_train.index.tolist(), "test_indices": X_test.index.tolist(),
    "model_parameters": {"max_iter": 2000, "random_state": SEED},
    "metrics": metrics, "parity": parity,
}
report_path = output_dir / "environment.json"
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(f"Saved run record: {report_path.relative_to(repo_root)}")